# A2.4 · Just-in-time authority

**Function A — AI Architecture, Risks and Mitigations → Controls — Identity and Ingress**  ·  *Security of AI*

Builds on **[A2.3 · Delegation that narrows, and survives audit](https://spbreed.github.io/cyber-commons/lessons/A2.3.html)**.

| | |
|---|---|
| Open-source tooling | Keycloak, OPA |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The hook

Standing authority means a successful injection always finds a live credential waiting. Just-in-time authority means the attacker has to arrive during the ninety seconds the grant exists, and be doing the one task it was scoped to.

## 2 · The framework

```
   standing                       just-in-time
   +-----------------+            +---------------------+
   | granted once    |            | granted per task    |
   | lives forever   |            | expires in 90s      |
   | any task        |            | this resource only  |
   +-----------------+            +---------------------+
   injection always finds         injection has to arrive
   a live credential              during the window, on that task
```

**Mitigates: T3 Privilege Compromise · T2 Tool Misuse.**

A2.3 narrows authority at the moment of delegation. This lesson removes it when
the task is over.

Standing authority is the reason an injection is always worth attempting: the
credential is there, permanently, waiting. Every successful A1.3 lands on a live
grant. Just-in-time authority changes the arithmetic — the attacker has to
arrive during a window that exists only while a specific task is running, and
that is bound to a specific resource.

Three properties, and the third is the one usually skipped:

**Short-lived.** Minutes, not months. Theft has a deadline.

**Purpose-bound.** Scoped to *this* resource, not to the resource class. Not
`reports:write` but `reports:write` on report 8812.

**Revoked on completion.** The grant ends when the task ends, not when the timer
does. A task that finishes in ten seconds should not leave a fifteen-minute
credential lying around, which is the difference between a TTL and an actual
lifecycle.

The operational cost is real and worth stating plainly: something must issue
these, and if that path breaks, work stops. That is the trade — a system that
fails closed under a control outage, in exchange for a system that has no
standing authority to steal.

> **What this control closes.**
>
> Removes the **standing** grant an injection needs. The attacker must now arrive inside a window bound to one task and one resource.

## 2 · The control

In [ ]:
GRANTS = {}
CLOCK = {"now": 1000}

def grant(task_id, principal, scope, resource, ttl=120):
    """Purpose-bound: this scope, on this resource, for this task."""
    GRANTS[task_id] = {"principal": principal, "scope": scope, "resource": resource,
                       "expires": CLOCK["now"] + ttl, "open": True}
    return task_id

def use(task_id, scope, resource):
    g = GRANTS.get(task_id)
    if not g:                                return False, "no such grant"
    if not g["open"]:                        return False, "task closed"
    if CLOCK["now"] > g["expires"]:          return False, "expired"
    if scope != g["scope"]:                  return False, f"scoped to {g['scope']}"
    if resource != g["resource"]:            return False, f"bound to {g['resource']}"
    return True, "permitted"

def close(task_id):
    if task_id in GRANTS:
        GRANTS[task_id]["open"] = False       # revoked on completion, not on expiry

grant("t-1", "dana@corp", "reports:write", "report/8812")

attempts = [
 ("the task's own write",        "reports:write", "report/8812"),
 ("a different report",          "reports:write", "report/9999"),
 ("a different scope",           "db:admin",      "report/8812"),
]
for label, scope, resource in attempts:
    ok, why = use("t-1", scope, resource)
    print(f"   {label:26s}{'ok' if ok else 'REFUSED':8s}{why}")

close("t-1")
ok, why = use("t-1", "reports:write", "report/8812")
print(f"   {'after the task completes':26s}{'ok' if ok else 'REFUSED':8s}{why}")

CLOCK["now"] = 2000
GRANTS["t-2"] = dict(GRANTS["t-1"], open=True, expires=1500)
ok, why = use("t-2", "reports:write", "report/8812")
print(f"   {'after the TTL expires':26s}{'ok' if ok else 'REFUSED':8s}{why}")
print()
print("An injection landing at 09:14 needs a task to be open, on the resource")
print("it wants, holding the scope it wants. Standing authority required none")
print("of those three things to line up.")
assert use("t-1", "reports:write", "report/8812")[0] is False

## What you just proved

A grant bound to one scope, one resource and one task permits only the task's own write — refusing a different report, a different scope, any use after the task closes, and any use after the TTL expires.

## Your turn

Take one standing grant an agent holds and work out what would break if it expired in two minutes. That list is the real cost of just-in-time, and it is usually shorter than expected.

---

**Next → [A2.5 · The non-human identity lifecycle](https://spbreed.github.io/cyber-commons/lessons/A2.5.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A2.4.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A2.4.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*